In [12]:
import asyncio
import httpx

from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client


async def main():
    url = "http://127.0.0.1:8001/mcp"

    async with httpx.AsyncClient(
        headers={
            "Authorization": "Bearer 123456789",
        },
        timeout=30.0,
    ) as http_client:

        async with streamable_http_client(
            url,
            http_client=http_client,
        ) as (read_stream, write_stream, get_session_id):

            async with ClientSession(
                read_stream,
                write_stream,
            ) as session:

                # =========================
                # INITIALIZE
                # =========================
                server = await session.initialize()

                print("Server:", server.serverInfo)
                print("Protocol:", server.protocolVersion)

                # =========================
                # GET TOOLS
                # =========================
                tools_result = await session.list_tools()

                print("\n" + "=" * 60)
                print("TOOLS")
                print("=" * 60)

                for tool in tools_result.tools:
                    print(f"\n[{tool.name}]")
                    print("Description:", tool.description)
                    print("Input schema:")
                    print(tool.inputSchema)

                # =========================
                # GET RESOURCES
                # =========================
                resources_result = await session.list_resources()

                print("\n" + "=" * 60)
                print("RESOURCES")
                print("=" * 60)

                for resource in resources_result.resources:
                    print(f"\n[{resource.name}]")
                    print("URI:", resource.uri)
                    print("Description:", resource.description)
                    print("MIME:", resource.mimeType)

                # =========================
                # GET RESOURCE TEMPLATES
                # =========================
                templates_result = await session.list_resource_templates()

                print("\n" + "=" * 60)
                print("RESOURCE TEMPLATES")
                print("=" * 60)

                for template in templates_result.resourceTemplates:
                    print(f"\n[{template.name}]")
                    print("URI:", template.uriTemplate)
                    print("Description:", template.description)

                # =========================
                # READ SKILLS
                # =========================
                print("\n" + "=" * 60)
                print("SKILLS")
                print("=" * 60)

                for resource in resources_result.resources:

                    # Ví dụ: chỉ đọc resource có vẻ là skill
                    if "skill" in str(resource.uri).lower():
                        print(f"\n--- {resource.uri} ---")

                        content = await session.read_resource(
                            resource.uri
                        )

                        for item in content.contents:
                            if hasattr(item, "text"):
                                print(item.text)
                            else:
                                print(item)




In [13]:
await main()

Server: name='stock-market-knowledge' title=None version='1.30.0' websiteUrl=None icons=None
Protocol: 2025-11-25

TOOLS

[evaluate_stock]
Description: Chạy toàn bộ pipeline đánh giá một mã cổ phiếu (hoặc chỉ số) theo các trường phái: giá trị (Buffett), tăng trưởng (CANSLIM), kỹ thuật (MA/RSI/MACD/Wyckoff/Elliott), kèm kế hoạch quản trị rủi ro (position sizing, R:R). Dùng cho mọi yêu cầu 'đánh giá/phân tích mã X'.
Input schema:
{'properties': {'symbol': {'title': 'Symbol', 'type': 'string'}, 'style': {'default': 'auto', 'title': 'Style', 'type': 'string'}, 'capital': {'default': 100000000, 'title': 'Capital', 'type': 'number'}, 'risk_pct': {'default': 1.5, 'title': 'Risk Pct', 'type': 'number'}, 'years': {'default': 3, 'title': 'Years', 'type': 'integer'}}, 'required': ['symbol'], 'title': 'evaluate_stockArguments', 'type': 'object'}

[get_history]
Description: Lấy dữ liệu OHLCV (open/high/low/close/volume) theo ngày cho một mã cổ phiếu hoặc chỉ số. Cổ phiếu VN dùng vnstock, chỉ số/mã 

In [ ]:
import asyncio
import httpx

from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client


async def main():

    async with httpx.AsyncClient(
        headers={
            "Authorization": "Bearer 123456789",
        }
    ) as http_client:

        async with streamable_http_client(
            "http://127.0.0.1:8001/mcp",
            http_client=http_client,
        ) as (read_stream, write_stream, get_session_id):

            async with ClientSession(
                read_stream,
                write_stream,
            ) as session:

                await session.initialize()

                # Liệt kê resources
                result = await session.list_resources()

                for resource in result.resources:
                    print(resource.uri)

                # Đọc KB resource
                content = await session.read_resource(
                    "kb://strategy-02-breakout-volume"
                )

                print("\n=== KB ===")

                for item in content.contents:
                    if hasattr(item, "text"):
                        print(item.text)
                        
await main()

kb://index
kb://01-basics
kb://02-buffett
kb://03-canslim
kb://04-wyckoff
kb://05-elliott
kb://06-risk
kb://strategy-01-ema-crossover
kb://strategy-02-breakout-volume
kb://strategy-03-breakout-retest
kb://strategy-04-ema-pullback
kb://strategy-05-rsi-bollinger-mr
kb://strategy-06-bb-squeeze
kb://strategy-07-vwap-pullback
kb://strategy-08-macd-ema200
kb://strategy-09-orb
kb://strategy-10-donchian-bo
kb://skill-danh-gia-co-phieu
kb://skill-chien-luoc-giao-dich

=== KB ===
# 02. Resistance/Support Breakout + Volume

> Pine Script v6 · Breakout · Volume confirmation.

---

## 1. Mục đích

Bắt **breakout khỏi vùng tích lũy** (range/consolidation) khi:

1. Giá phá **resistance** (hoặc **support**) đã hình thành trong N nến trước.
2. Khối lượng **đột biến** vượt ngưỡng trung bình → xác nhận "tiền thật" đẩy giá, không phải nhiễu.

Volume filter là chìa khoá phân biệt breakout thật với fakeout.

---

## 2. Khi nào hoạt động tốt / xấu

### Phù hợp

- Sau một giai đoạn **sideway rõ rệt** (range h